# 02 - Pressure Meter
Innings 2 (chase): leverage = expected swing in win probability on the next ball.
Innings 1: collapse / rebuild state = projected final score vs the par for that match (no target exists).
Both are turned into a 0-100 percentile WITHIN competition tier, separately for batters (p_bat) and bowlers (p_bowl).
All models are out-of-fold (grouped by match), so no ball is scored by a model that saw its match.
Output: outputs/balls_pressure.parquet

In [1]:
import warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220, 'display.max_columns', 30)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
OUT = ROOT / 'outputs'
b = pd.read_parquet(OUT / 'balls_features.parquet')

TIER = {'IPL': 0, 'FM': 1, 'ASSOC': 2, 'A': 3}
GRP = {'IPL': 0, 'T20I': 1, 'A_TEAM': 2}
b['g_c'] = b.comp_group.map(GRP)
b['bt_c'] = b.bat_tier.map(TIER)
b['ot_c'] = b.bowl_tier.map(TIER)

CAT = ['g_c', 'bt_c', 'ot_c']
FEATS = {1: ['score', 'wkts', 'balls_left', 'crr'] + CAT,
         2: ['runs_needed', 'balls_left', 'wkts', 'rrr', 'crr'] + CAT}
MONO = {1: {'score': 1, 'wkts': -1}, 2: {'runs_needed': -1, 'wkts': -1, 'rrr': -1}}


def make(inn):
    f = FEATS[inn]
    return HGB(max_iter=300, learning_rate=0.05, max_leaf_nodes=20, min_samples_leaf=300, l2_regularization=1.0,
               categorical_features=[f.index(c) for c in CAT], monotonic_cst=[MONO[inn].get(x, 0) for x in f],
               random_state=0)

## Grouped 5-fold by match

In [2]:
R = b[b.innings.isin([1, 2])].copy()
train_matches = R.loc[R.train_ok, 'match_id'].unique()
perm = np.random.default_rng(42).permutation(train_matches)
R['fold'] = R.match_id.map({mid: i % 5 for i, mid in enumerate(perm)}).fillna(-1).astype(int)

models = {}
for inn in (1, 2):
    for f in range(5):
        tr = R[(R.innings == inn) & R.train_ok & (R.fold != f)]
        models[(inn, f)] = make(inn).fit(tr[FEATS[inn]], tr.bat_win.astype(int))


def wp(inn, X, fold):
    """Out-of-fold win prob for the batting side. fold=-1 rows (never trained on) use the 5-model average."""
    out = np.zeros(len(X))
    for f in range(5):
        i = fold == f
        if i.any():
            out[i] = models[(inn, f)].predict_proba(X[i])[:, 1]
    i = fold == -1
    if i.any():
        out[i] = np.mean([models[(inn, f)].predict_proba(X[i])[:, 1] for f in range(5)], axis=0)
    return out


R['wp'] = np.nan
for inn in (1, 2):
    i = R.innings == inn
    R.loc[i, 'wp'] = wp(inn, R.loc[i, FEATS[inn]], R.loc[i, 'fold'].values)

## Model quality (out-of-fold)

In [3]:
rows = []
for inn in (1, 2):
    d = R[(R.innings == inn) & R.train_ok]
    y = d.bat_win.astype(int)
    rows.append({'innings': inn, 'balls': len(d), 'logloss': log_loss(y, d.wp), 'baseline_logloss': log_loss(y, np.full(len(d), y.mean())),
                 'brier': brier_score_loss(y, d.wp), 'auc': roc_auc_score(y, d.wp)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
d = R[R.train_ok & (R.innings == 2)]
cal = d.groupby(pd.cut(d.wp, [0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1])).agg(n=('wp', 'size'), predicted=('wp', 'mean'), actual=('bat_win', 'mean'))
print('\ncalibration, chase innings'); print(cal.round(3))

 innings  balls  logloss  baseline_logloss  brier   auc
       1 101363    0.645             0.693  0.222 0.698
       2  93030    0.406             0.692  0.130 0.899

calibration, chase innings
                n  predicted  actual
wp                                  
(0.0, 0.1]  23591      0.028   0.048
(0.1, 0.2]   8126      0.147   0.194
(0.2, 0.3]   6190      0.249   0.272
(0.3, 0.4]   6105      0.349   0.373
(0.4, 0.5]   5321      0.448   0.437
(0.5, 0.6]   4761      0.550   0.524
(0.6, 0.7]   5102      0.650   0.645
(0.7, 0.8]   5643      0.753   0.771
(0.8, 0.9]   7835      0.854   0.785
(0.9, 1.0]  20356      0.973   0.942


## Leverage: expected swing in win probability on the next ball
Next-ball outcomes (0,1,2,3,4,6,W) are weighted by their real frequency in that innings and phase.

In [4]:
KEYS = ['0', '1', '2', '3', '4', '6', 'W']
leg = R[R.legal]
oc = np.where(leg.is_wicket, 'W', leg.tot.astype(str))
PR = pd.crosstab([leg.innings, leg.phase], oc, normalize='index')[KEYS]
PR = PR.div(PR.sum(axis=1), axis=0)  # drop 5s / 7+ and renormalise

R['wp_exp'] = np.nan
R['lev'] = np.nan
for inn in (1, 2):
    d = R[(R.innings == inn) & (R.balls_left >= 1)]
    fold = d.fold.values
    P = d[['phase']].merge(PR.loc[inn], left_on='phase', right_index=True, how='left')[KEYS].values
    W = np.zeros((len(d), len(KEYS)))
    for j, k in enumerate(KEYS):
        r, wk = (0, 1) if k == 'W' else (int(k), 0)
        n = pd.DataFrame(index=d.index)
        n['score'] = d.score + r
        n['wkts'] = d.wkts + wk
        n['balls_left'] = d.balls_left - 1
        n['crr'] = n.score / (d.lb + 1) * 6
        n[CAT] = d[CAT]
        if inn == 2:
            n['runs_needed'] = d.runs_needed - r
            n['rrr'] = np.where(n.balls_left > 0, n.runs_needed / n.balls_left.clip(lower=1) * 6, np.nan)
            v = wp(2, n[FEATS[2]], fold)
            win = n.runs_needed <= 0
            lose = ((n.wkts >= 10) | (n.balls_left <= 0)) & ~win
            v = np.where(win, 1.0, np.where(lose, np.where(n.runs_needed == 1, 0.5, 0.0), v))
        else:
            v = wp(1, n[FEATS[1]], fold)
            t = ((n.wkts >= 10) | (n.balls_left <= 0)).values
            if t.any():  # innings ends: value = 1 - chasing side's win prob at the start of the chase
                c = pd.DataFrame(index=n.index[t])
                c['runs_needed'] = n.score[t] + 1
                c['balls_left'] = d.max_balls[t]
                c['wkts'] = 0
                c['rrr'] = c.runs_needed / c.balls_left * 6
                c['crr'] = 0.0
                c['g_c'], c['bt_c'], c['ot_c'] = d.g_c[t], d.ot_c[t], d.bt_c[t]
                v[t] = 1 - wp(2, c[FEATS[2]], fold[t])
        W[:, j] = v
    exp = (P * W).sum(1)
    R.loc[d.index, 'wp_exp'] = exp
    R.loc[d.index, 'lev'] = (P * np.abs(W - exp[:, None])).sum(1)

## Innings 1: projected final score vs par
par = projected total at ball 1 for that match (depends on tier). gap = projection now - par.
gap < 0: batting side behind par (batting pressure). gap > 0: batting side ahead (bowling pressure).

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGR

F1 = FEATS[1]
final = R[R.innings == 1].groupby('match_id').tot.sum()
R['final'] = R.match_id.map(final).where(R.innings == 1)


def make_reg():
    return HGR(max_iter=300, learning_rate=0.05, max_leaf_nodes=20, min_samples_leaf=300, l2_regularization=1.0,
               categorical_features=[F1.index(c) for c in CAT],
               monotonic_cst=[{'score': 1, 'wkts': -1, 'balls_left': 1}.get(x, 0) for x in F1], random_state=0)


reg_models = {f: make_reg().fit(R[(R.innings == 1) & R.train_ok & (R.fold != f)][F1],
                                R[(R.innings == 1) & R.train_ok & (R.fold != f)].final) for f in range(5)}


def proj(X, fold):
    out = np.zeros(len(X))
    for f in range(5):
        i = fold == f
        if i.any():
            out[i] = reg_models[f].predict(X[i])
    i = fold == -1
    if i.any():
        out[i] = np.mean([reg_models[f].predict(X[i]) for f in range(5)], axis=0)
    return out


I = R.innings == 1
R.loc[I, 'proj'] = proj(R.loc[I, F1], R.loc[I, 'fold'].values)
S = R.loc[I].copy()
S['score'], S['wkts'], S['balls_left'], S['crr'] = 0, 0, S.max_balls, 0.0
R.loc[I, 'par_start'] = proj(S[F1], S.fold.values)
R['gap'] = R.proj - R.par_start

# runs lost when a wicket falls on this ball (Duckworth-Lewis style resource value, from the same projection model)
rr = R.balls_left >= 1
Xn = R[rr].copy()
Xn['balls_left'] = Xn.balls_left - 1
Xn['crr'] = Xn.score / (Xn.lb + 1) * 6
Xw = Xn.copy()
Xw['wkts'] = np.minimum(Xn.wkts + 1, 10)
R.loc[rr, 'wkt_cost'] = proj(Xn[F1], Xn.fold.values) - proj(Xw[F1], Xw.fold.values)
print('\nWicket cost (runs) by phase:'); print(R.groupby('phase').wkt_cost.mean().round(1).to_dict())

d = R[I & R.train_ok]
rows = []
for o in (3, 6, 10, 15):
    x = d[(d.overid == o) & (d.over_ball_num == 1)]
    rows.append({'over': o, 'MAE_model': (x.proj - x.final).abs().mean(), 'MAE_league_mean': (x.final - x.final.mean()).abs().mean()})
print('Projection accuracy (out-of-fold, runs)'); print(pd.DataFrame(rows).round(1).to_string(index=False))


Wicket cost (runs) by phase:
{'DEATH': 2.5, 'MID': 5.8, 'PP': 9.7}
Projection accuracy (out-of-fold, runs)
 over  MAE_model  MAE_league_mean
    3       26.2             29.2
    6       22.6             29.2
   10       18.5             29.2
   15       12.6             29.0


## Percentiles within tier

In [6]:
R['match_tier'] = np.select([R.comp_group.eq('IPL'), R.comp_group.eq('A_TEAM'), R.bat_tier.eq('FM') & R.bowl_tier.eq('FM')],
                            ['IPL', 'A_TEAM', 'T20I_FM'], 'T20I_ASSOC')


def pct(s, g):
    return s.groupby(g).rank(pct=True) * 100


R['p_bat'] = np.nan
R['p_bowl'] = np.nan
i2 = (R.innings == 2) & R.lev.notna()
R.loc[i2, 'p_bat'] = pct(R.loc[i2, 'lev'], R.loc[i2, 'match_tier'])
R.loc[i2, 'p_bowl'] = R.loc[i2, 'p_bat']
i1 = (R.innings == 1) & R.gap.notna()
R.loc[i1, 'p_bat'] = pct(-R.loc[i1, 'gap'], R.loc[i1, 'match_tier'])
R.loc[i1, 'p_bowl'] = pct(R.loc[i1, 'gap'], R.loc[i1, 'match_tier'])

## Merge back; super overs = maximum pressure by definition

In [7]:
COLS = ['match_id', 'innings', 'innings_ball_num', 'wp', 'wp_exp', 'lev', 'proj', 'par_start', 'gap', 'wkt_cost', 'p_bat', 'p_bowl']
b = b.merge(R[COLS], on=['match_id', 'innings', 'innings_ball_num'], how='left')
b['match_tier'] = b.match_id.map(R.drop_duplicates('match_id').set_index('match_id').match_tier)
b.loc[b.is_super_over, ['p_bat', 'p_bowl']] = 100.0
LAB = ['Low', 'Medium', 'High', 'Extreme']
BINS = [0, 50, 80, 95, 100.01]
b['bucket_bat'] = pd.cut(b.p_bat, BINS, labels=LAB, include_lowest=True)
b['bucket_bowl'] = pd.cut(b.p_bowl, BINS, labels=LAB, include_lowest=True)
b.to_parquet(OUT / 'balls_pressure.parquet')

## Evidence: do pressure balls differ from ordinary balls?

In [8]:
def summ(x):
    l = x[x.legal]
    return pd.Series({'balls': len(l), 'runs/over': l.tot.sum() / len(l) * 6, 'dot%': (l.tot == 0).mean() * 100,
                      'boundary%': (l.is_four | l.is_six).mean() * 100, 'wkts/100b': l.is_wicket.mean() * 100})


reg = b[b.innings.isin([1, 2]) & b.p_bat.notna()]
print('\nBATTING view: outcomes by batting-pressure bucket'); print(reg.groupby(['innings', 'bucket_bat'], observed=True).apply(summ).round(2))
print('\nBOWLING view: outcomes by bowling-pressure bucket'); print(reg.groupby(['innings', 'bucket_bowl'], observed=True).apply(summ).round(2))
print('\nShare of balls with pressure >= 80, by innings x phase')
print(reg.groupby(['innings', 'phase']).agg(bat=('p_bat', lambda s: (s >= 80).mean()), bowl=('p_bowl', lambda s: (s >= 80).mean())).round(3))

c1 = reg[(reg.innings == 1) & reg.overid.between(7, 12)]
print('\nInnings 1, overs 7-12: collapse state check')
print(c1.groupby(pd.cut(c1.wkts, [-1, 1, 3, 10], labels=['0-1 down', '2-3 down', '4+ down']), observed=True).agg(
    balls=('p_bat', 'size'), mean_gap_runs=('gap', 'mean'), mean_p_bat=('p_bat', 'mean'), runs_per_over=('tot', lambda s: s.sum() / len(s) * 6)).round(2))

c2 = reg[reg.innings == 2]
print('\nChase: pressure by required rate')
print(c2.groupby(pd.cut(c2.rrr, [-np.inf, 6, 8, 10, 12, 15, 20, np.inf]), observed=True).agg(
    balls=('p_bat', 'size'), mean_pressure=('p_bat', 'mean'), mean_winprob=('wp', 'mean')).round(2))
print('\nBalls by match tier:'); print(reg.match_tier.value_counts())
print('saved', OUT / 'balls_pressure.parquet')


BATTING view: outcomes by batting-pressure bucket
                      balls  runs/over   dot%  boundary%  wkts/100b
innings bucket_bat                                                 
1       Low         50332.0       9.42  31.14      21.13       5.77
        Medium      30498.0       8.11  37.39      17.65       5.01
        High        15275.0       7.51  37.26      14.85       5.79
        Extreme      5085.0       6.38  44.76      11.86       5.62
2       Low         45693.0       8.48  35.37      18.43       6.07
        Medium      27382.0       8.28  37.26      18.66       4.97
        High        13654.0       8.38  37.14      19.03       5.38
        Extreme      4493.0       8.87  34.19      19.50       7.86

BOWLING view: outcomes by bowling-pressure bucket
                       balls  runs/over   dot%  boundary%  wkts/100b
innings bucket_bowl                                                 
1       Low          50853.0       7.76  38.08      16.23       5.30
        Med